# Huấn luyện LightGBM

**Trước khi chạy:** Settings → Accelerator: **None (CPU)**. Input phải có dataset dữ liệu sạch (unified_clean.parquet).

1. Để `QUICK = True`, bấm **Run All** để chạy thử (~10–20 phút). Số liệu lần này KHÔNG dùng.
2. Không lỗi thì sửa `QUICK = False` → **Save Version → Save & Run All** (chạy nền, vài giờ).

Ba cell đầu chỉ **ghi file** dùng chung ra /kaggle/working (dòng `%%writefile`), không huấn luyện gì. Không sửa nội dung ba cell này.

In [ ]:
%%writefile ids_features.py
"""ids_features.py — tiền xử lý đặc trưng và luật quyết định DÙNG CHUNG.

File này được dùng ở CẢ BA nơi, không được viết lại ở đâu khác:
  1. notebook huấn luyện (Kaggle),
  2. đánh giá trên dữ liệu lab,
  3. backend FastAPI.
Chỉ phụ thuộc numpy + pandas để backend không phải cài thêm gì.
"""
import numpy as np
import pandas as pd

# Tên cột kiểu CSE-CIC-IDS2018 / CICFlowMeter-V4 -> tên chuẩn (theo CIC-IDS2017).
# CICFlowMeter bản Java mới xuất ra tên kiểu 2018, nên backend cần bảng này.
RENAME_2018 = {
    'ACK Flag Cnt': 'ACK Flag Count', 'Bwd Blk Rate Avg': 'Bwd Avg Bulk Rate',
    'Bwd Byts/b Avg': 'Bwd Avg Bytes/Bulk', 'Bwd Header Len': 'Bwd Header Length',
    'Bwd IAT Tot': 'Bwd IAT Total', 'Bwd Pkt Len Max': 'Bwd Packet Length Max',
    'Bwd Pkt Len Mean': 'Bwd Packet Length Mean', 'Bwd Pkt Len Min': 'Bwd Packet Length Min',
    'Bwd Pkt Len Std': 'Bwd Packet Length Std', 'Bwd Pkts/b Avg': 'Bwd Avg Packets/Bulk',
    'Bwd Pkts/s': 'Bwd Packets/s', 'Bwd Seg Size Avg': 'Avg Bwd Segment Size',
    'Dst Port': 'Destination Port', 'ECE Flag Cnt': 'ECE Flag Count',
    'FIN Flag Cnt': 'FIN Flag Count', 'Flow Byts/s': 'Flow Bytes/s',
    'Flow Pkts/s': 'Flow Packets/s', 'Fwd Act Data Pkts': 'act_data_pkt_fwd',
    'Fwd Blk Rate Avg': 'Fwd Avg Bulk Rate', 'Fwd Byts/b Avg': 'Fwd Avg Bytes/Bulk',
    'Fwd Header Len': 'Fwd Header Length', 'Fwd IAT Tot': 'Fwd IAT Total',
    'Fwd Pkt Len Max': 'Fwd Packet Length Max', 'Fwd Pkt Len Mean': 'Fwd Packet Length Mean',
    'Fwd Pkt Len Min': 'Fwd Packet Length Min', 'Fwd Pkt Len Std': 'Fwd Packet Length Std',
    'Fwd Pkts/b Avg': 'Fwd Avg Packets/Bulk', 'Fwd Pkts/s': 'Fwd Packets/s',
    'Fwd Seg Size Avg': 'Avg Fwd Segment Size', 'Fwd Seg Size Min': 'min_seg_size_forward',
    'Init Bwd Win Byts': 'Init_Win_bytes_backward', 'Init Fwd Win Byts': 'Init_Win_bytes_forward',
    'PSH Flag Cnt': 'PSH Flag Count', 'Pkt Len Max': 'Max Packet Length',
    'Pkt Len Mean': 'Packet Length Mean', 'Pkt Len Min': 'Min Packet Length',
    'Pkt Len Std': 'Packet Length Std', 'Pkt Len Var': 'Packet Length Variance',
    'Pkt Size Avg': 'Average Packet Size', 'RST Flag Cnt': 'RST Flag Count',
    'SYN Flag Cnt': 'SYN Flag Count', 'Subflow Bwd Byts': 'Subflow Bwd Bytes',
    'Subflow Bwd Pkts': 'Subflow Bwd Packets', 'Subflow Fwd Byts': 'Subflow Fwd Bytes',
    'Subflow Fwd Pkts': 'Subflow Fwd Packets', 'Tot Bwd Pkts': 'Total Backward Packets',
    'Tot Fwd Pkts': 'Total Fwd Packets', 'TotLen Bwd Pkts': 'Total Length of Bwd Packets',
    'TotLen Fwd Pkts': 'Total Length of Fwd Packets', 'URG Flag Cnt': 'URG Flag Count',
}


def port_class(port):
    """0: cổng hệ thống (<1024), 1: cổng đăng ký (1024-49151), 2: cổng tạm (>=49152)."""
    p = pd.to_numeric(pd.Series(port), errors='coerce').to_numpy(np.float64)
    out = np.where(p < 1024, 0.0, np.where(p < 49152, 1.0, 2.0))
    out[np.isnan(p)] = np.nan
    return out


def canonicalize(df_raw: pd.DataFrame) -> pd.DataFrame:
    """Chuẩn hoá tên cột (bỏ khoảng trắng, đổi tên kiểu 2018) và suy ra dst_port_class."""
    df = df_raw.rename(columns=lambda c: str(c).strip()).rename(columns=RENAME_2018)
    if 'dst_port_class' not in df.columns and 'Destination Port' in df.columns:
        df['dst_port_class'] = port_class(df['Destination Port'])
    return df


def prepare_features(df_raw: pd.DataFrame, features: list):
    """Trả về (X float32 đúng thứ tự `features`, valid: mask dòng dùng được).

    Dòng không hợp lệ (NaN/inf, thời lượng <= 0) KHÔNG có trong dữ liệu huấn luyện,
    nên không được đưa vào mô hình: backend đánh dấu riêng các dòng này.
    """
    df = canonicalize(df_raw)
    missing = [c for c in features if c not in df.columns]
    if missing:
        raise KeyError(f'Thiếu {len(missing)} cột đặc trưng: {missing}')
    X = df[features].apply(pd.to_numeric, errors='coerce').to_numpy(np.float32)
    valid = np.isfinite(X).all(axis=1)
    if 'Flow Duration' in df.columns:
        valid &= pd.to_numeric(df['Flow Duration'], errors='coerce').to_numpy() > 0
    return X, valid


def decide(proba: np.ndarray, benign_index: int, threshold=None) -> np.ndarray:
    """Luật quyết định duy nhất cho mọi nơi.

    threshold=None : lớp có xác suất cao nhất (argmax).
    threshold=t    : là tấn công nếu 1 - P(BENIGN) >= t; khi đó chọn lớp tấn công
                     có xác suất cao nhất. Ngưỡng t được chọn trên tập validation.
    """
    if threshold is None:
        return proba.argmax(axis=1)
    attack = proba.copy()
    attack[:, benign_index] = -1.0
    return np.where(1.0 - proba[:, benign_index] >= threshold,
                     attack.argmax(axis=1), benign_index)


def predict_frame(bundle: dict, df_raw: pd.DataFrame) -> pd.DataFrame:
    """Hàm backend gọi: nhận DataFrame thô từ CICFlowMeter, trả nhãn + điểm tấn công."""
    X, valid = prepare_features(df_raw, bundle['features'])
    out = pd.DataFrame({'valid': valid, 'label': None, 'attack_score': np.nan})
    if valid.any():
        model = bundle['model']
        p = np.zeros((int(valid.sum()), len(bundle['classes'])), np.float32)
        p[:, model.classes_] = model.predict_proba(X[valid])
        pred = decide(p, bundle['benign_index'], bundle['threshold'])
        out.loc[valid, 'label'] = np.asarray(bundle['classes'], dtype=object)[pred]
        out.loc[valid, 'attack_score'] = 1.0 - p[:, bundle['benign_index']]
    return out


In [ ]:
%%writefile ids_common.py
"""ids_common.py — phần dùng chung cho MỌI mô hình (RF, XGBoost, LightGBM, MLP, IF).

Mọi notebook huấn luyện import file này để:
  - đọc cùng một bộ dữ liệu, cùng cách chia fold (không ai tự chia lại),
  - đánh giá bằng cùng một hàm, cùng một bộ chỉ số,
  - ghi cùng một nhật ký thí nghiệm (experiments.csv),
  - đóng gói mô hình theo cùng một định dạng cho backend.
"""
import json, os, platform, time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             precision_recall_fscore_support, roc_auc_score)

from ids_features import decide, prepare_features

WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.').resolve()
LOG_PATH = WORK / 'experiments.csv'
PROTOCOL = ('dedup vector trùng (F_ALL) + 5 fold theo nhóm (hash F_NOOS), '
            'phân tầng bộ dữ liệu×lớp, test = fold 0, seed 42')


# ----------------------------------------------------------------------------- dữ liệu
def load_data(data_dir=None):
    """Đọc parquet sạch + feature_sets.json. Tự tìm trong /kaggle/input nếu không truyền."""
    if data_dir is None:
        data_dir = os.environ.get('IDS_DATA_DIR')
    if data_dir is None:
        hits = sorted(Path('/kaggle/input').rglob('unified_clean.parquet'))
        assert len(hits) == 1, f'Cần đúng 1 file unified_clean.parquet, tìm thấy: {hits}'
        data_dir = hits[0].parent
    data_dir = Path(data_dir)
    df = pd.read_parquet(data_dir / 'unified_clean.parquet')
    fs = json.load(open(data_dir / 'feature_sets.json', encoding='utf-8'))
    sha = None
    rp = data_dir / 'clean_report.json'
    if rp.exists():
        sha = json.load(open(rp, encoding='utf-8')).get('ket_qua', {}).get('sha256')
    df['y'] = pd.Categorical(df['label'], categories=fs['classes']).codes.astype(np.int8)
    assert (df['y'] >= 0).all(), 'Có nhãn ngoài danh sách classes'
    for key in ('F_ALL', 'F_NOOS'):
        assert all(c in df.columns for c in fs[key]), f'Thiếu cột của {key}'
    print(f'Dữ liệu: {len(df):,} dòng | F_ALL={len(fs["F_ALL"])} | F_NOOS={len(fs["F_NOOS"])} '
          f'| sha256={str(sha)[:12]}')
    return df, fs, sha


def cap_sample(df, cap, benign_cap=None, seed=42):
    """Lấy mẫu theo từng ô (bộ dữ liệu × lớp), mỗi ô tối đa `cap` dòng (BENIGN: benign_cap)."""
    parts = []
    for (_, lab), g in df.groupby(['source_dataset', 'label'], observed=True):
        c = benign_cap if (lab == 'BENIGN' and benign_cap) else cap
        parts.append(g if len(g) <= c else g.sample(c, random_state=seed))
    return pd.concat(parts)


def frac_sample(df, frac, min_rows=300, seed=42):
    """Lấy mẫu một tỉ lệ ở mỗi ô nhưng giữ tối thiểu min_rows (dùng cho chế độ QUICK)."""
    parts = []
    for _, g in df.groupby(['source_dataset', 'label', 'fold'], observed=True):
        n = min(len(g), max(min_rows, int(len(g) * frac)))
        parts.append(g.sample(n, random_state=seed))
    return pd.concat(parts)


def xy(df, feats):
    return df[feats].to_numpy(np.float32), df['y'].to_numpy()


def full_proba(model, X, k):
    """predict_proba luôn trả đủ k cột, kể cả khi mô hình chỉ học một phần các lớp."""
    p = np.zeros((len(X), k), np.float32)
    p[:, model.classes_] = model.predict_proba(X)
    return p


# ----------------------------------------------------------------------------- đánh giá
def evaluate(y, proba, pred, classes, meta=None, weights=None):
    """Trả (chỉ số dạng số, các bảng chi tiết). Macro chỉ tính trên các lớp có trong y."""
    y, pred = np.asarray(y), np.asarray(pred)
    b = classes.index('BENIGN')
    present = [int(c) for c in np.unique(y)]
    s = {'n': int(len(y)),
         'macro_f1': f1_score(y, pred, labels=present, average='macro', zero_division=0)}
    if weights is not None:
        s['macro_f1_w_ndup'] = f1_score(y, pred, labels=present, average='macro',
                                        sample_weight=np.asarray(weights), zero_division=0)
    p, r, f, sup = precision_recall_fscore_support(y, pred, labels=present, zero_division=0)
    for i, c in enumerate(present):
        name = classes[c]
        s[f'precision_{name}'], s[f'recall_{name}'], s[f'f1_{name}'] = p[i], r[i], f[i]
        s[f'support_{name}'] = int(sup[i])
        yc = (y == c)
        if 0 < yc.sum() < len(y):
            s[f'auc_{name}'] = roc_auc_score(yc, proba[:, c])
            s[f'prauc_{name}'] = average_precision_score(yc, proba[:, c])
    aucs = [s[f'auc_{classes[c]}'] for c in present if f'auc_{classes[c]}' in s]
    s['auc_ovr_macro'] = float(np.mean(aucs)) if aucs else np.nan
    is_b = (y == b)
    s['fpr_benign'] = float((pred[is_b] != b).mean()) if is_b.any() else np.nan
    s['detection_rate'] = float((pred[~is_b] != b).mean()) if (~is_b).any() else np.nan
    if is_b.any() and (~is_b).any():
        s['auc_attack_vs_benign'] = roc_auc_score(~is_b, 1.0 - proba[:, b])
    s['accuracy_chi_tham_khao'] = float((pred == y).mean())

    t = {'confusion': pd.DataFrame(confusion_matrix(y, pred, labels=range(len(classes))),
                                   index=[f'that_{c}' for c in classes],
                                   columns=[f'du_doan_{c}' for c in classes])}
    if meta is not None:
        m = meta[['source_dataset', 'label', 'attack_subtype']].copy()
        m['dung'] = (pred == y)
        m['bao_tan_cong'] = (pred != b)
        t['theo_subtype'] = (m.groupby(['source_dataset', 'label', 'attack_subtype'], observed=True)
                             .agg(n=('dung', 'size'), ti_le_dung=('dung', 'mean'),
                                  ti_le_bao_tan_cong=('bao_tan_cong', 'mean'))
                             .round(4).reset_index())
    return {k: (round(float(v), 6) if isinstance(v, (float, np.floating)) else v)
            for k, v in s.items()}, t


def choose_threshold(y, proba, benign_index, classes, target_fpr=0.01):
    """Chọn ngưỡng trên VALIDATION: macro-F1 cao nhất với điều kiện FPR(BENIGN) <= target."""
    y = np.asarray(y)
    score = 1.0 - proba[:, benign_index]
    is_b = (y == benign_index)
    present = list(np.unique(y))
    cands = np.unique(np.concatenate([
        np.linspace(0.02, 0.98, 49),
        np.quantile(score[is_b], [0.99, 0.995, 0.999, 0.9995, 0.9999])]))
    rows = []
    for t in cands:
        pred = decide(proba, benign_index, t)
        rows.append({'threshold': float(t),
                     'fpr_benign': float((pred[is_b] != benign_index).mean()),
                     'macro_f1': f1_score(y, pred, labels=present, average='macro', zero_division=0)})
    tab = pd.DataFrame(rows)
    ok = tab[tab.fpr_benign <= target_fpr]
    if ok.empty:
        print(f'CẢNH BÁO: không ngưỡng nào đạt FPR <= {target_fpr}; chọn ngưỡng có FPR thấp nhất.')
        best = tab.sort_values(['fpr_benign', 'macro_f1'], ascending=[True, False]).iloc[0]
    else:
        best = ok.sort_values('macro_f1', ascending=False).iloc[0]
    return float(best.threshold), tab


def inference_time(model, X, batch=10_000, repeat=3):
    """Mili giây cho một lô `batch` flow và cho 1 flow đơn lẻ (trung vị của `repeat` lần)."""
    Xb = X[:batch]
    t_b, t_1 = [], []
    for _ in range(repeat):
        t0 = time.perf_counter(); model.predict_proba(Xb); t_b.append(time.perf_counter() - t0)
        t0 = time.perf_counter(); model.predict_proba(X[:1]); t_1.append(time.perf_counter() - t0)
    return {'ms_per_batch': round(1000 * float(np.median(t_b)), 1), 'batch_rows': len(Xb),
            'ms_per_1_flow': round(1000 * float(np.median(t_1)), 2)}


# ----------------------------------------------------------------------------- nhật ký
def versions():
    import scipy
    v = {'python': platform.python_version(), 'scikit-learn': sklearn.__version__,
         'numpy': np.__version__, 'pandas': pd.__version__, 'scipy': scipy.__version__,
         'joblib': joblib.__version__}
    for lib in ('xgboost', 'lightgbm'):       # chỉ ghi nếu notebook đã dùng thư viện đó
        import sys
        if lib in sys.modules:
            v[lib] = sys.modules[lib].__version__
    return v


def log_run(run_id, model_name, fset, stage, scores, params=None, sha=None, extra=None):
    """Thêm một dòng vào experiments.csv. Mỗi dòng = một lần đánh giá."""
    row = {'run_id': run_id, 'thoi_gian': time.strftime('%Y-%m-%d %H:%M:%S'),
           'mo_hinh': model_name, 'bo_dac_trung': fset, 'giai_doan': stage,
           'giao_thuc_chia': PROTOCOL, 'data_sha256': sha,
           'tham_so': json.dumps(params, ensure_ascii=False, default=str) if params else '',
           **scores, **(extra or {})}
    new = pd.DataFrame([row])
    old = pd.read_csv(LOG_PATH) if LOG_PATH.exists() else pd.DataFrame()
    pd.concat([old, new], ignore_index=True).to_csv(LOG_PATH, index=False, encoding='utf-8')
    keys = ['macro_f1', 'fpr_benign', 'detection_rate', 'auc_ovr_macro']
    print(f'  [{stage}] ' + ' | '.join(f'{k}={scores[k]:.4f}' for k in keys
                                         if k in scores and scores[k] == scores[k]))


def save_tables(run_dir, stage, tables):
    run_dir.mkdir(parents=True, exist_ok=True)
    for name, t in tables.items():
        t.to_csv(run_dir / f'{stage}_{name}.csv', encoding='utf-8')


# ----------------------------------------------------------------------------- đóng gói
def save_bundle(path, model, features, fset, classes, threshold, target_fpr,
                params, run_id, sha, test_scores, model_name):
    """Lưu MỘT file .joblib chứa đủ mọi thứ backend cần. Chỉ chứa đối tượng sklearn + dict,
    không có class tự viết, nên backend chỉ cần cài đúng phiên bản thư viện là load được."""
    bundle = {
        'model': model, 'model_name': model_name,
        'features': list(features), 'feature_set': fset,
        'classes': list(classes), 'benign_index': classes.index('BENIGN'),
        'threshold': threshold, 'target_fpr': target_fpr,
        'decision_rule': 'tấn công nếu 1-P(BENIGN) >= threshold; lớp = argmax các lớp tấn công',
        'preprocessing': 'ids_features.prepare_features (không scale)',
        'params': params, 'run_id': run_id, 'data_sha256': sha, 'split_protocol': PROTOCOL,
        'test_scores': test_scores, 'versions': versions(),
        'created': time.strftime('%Y-%m-%d %H:%M:%S'),
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(bundle, path, compress=3)
    card = {k: v for k, v in bundle.items() if k != 'model'}
    json.dump(card, open(path.with_suffix('.json'), 'w', encoding='utf-8'),
              ensure_ascii=False, indent=2, default=str)
    v = bundle['versions']
    (path.parent / f'requirements_backend_{path.stem}.txt').write_text(
        '\n'.join(f'{k}=={val}' for k, val in v.items() if k != 'python') + '\n',
        encoding='utf-8')
    return bundle


def evaluate_external(bundle, csv_path, label_col, label_map):
    """Đánh giá mô hình đã đóng gói trên CSV do CICFlowMeter xuất ra (dữ liệu lab)."""
    raw = pd.read_csv(csv_path)
    X, valid = prepare_features(raw, bundle['features'])
    lab = raw[label_col].astype(str).str.strip().map(label_map)
    unknown = sorted(raw.loc[lab.isna(), label_col].astype(str).unique())
    assert not unknown, f'Nhãn lab chưa có trong label_map: {unknown}'
    classes = bundle['classes']
    y = lab.map({c: i for i, c in enumerate(classes)}).to_numpy()
    print(f'Lab: {len(raw):,} dòng, loại {int((~valid).sum()):,} dòng không hợp lệ (NaN/inf/thời lượng 0)')
    model = bundle['model']
    p = full_proba(model, X[valid], len(classes))
    pred = decide(p, bundle['benign_index'], bundle['threshold'])
    return evaluate(y[valid], p, pred, classes)


In [ ]:
%%writefile ids_runner.py
"""ids_runner.py — quy trình huấn luyện/đánh giá CHUNG cho XGBoost, LightGBM, MLP.

Các giai đoạn và tên dòng trong experiments.csv giống hệt notebook Random Forest
(tune_cv4_mau, cv_fold{k}, cv4_trung_binh, test_argmax, test_nguong, cheo_test{bộ}, lab),
nên notebook so sánh đọc được kết quả của mọi mô hình như nhau.

Khác RF ở một điểm: các mô hình này cần early stopping. Tập early stopping là 10% NHÓM
(theo group_id) của phần huấn luyện, tách khỏi fold đang đánh giá, nên điểm CV và điểm
test không bị lạc quan do chọn số vòng/epoch trên chính tập được chấm điểm.

Mỗi mô hình chỉ cần cung cấp một "adapter":
    adapter.name                      tên mô hình (ghi vào log)
    adapter.short                     tên ngắn cho file (xgb, lgbm, mlp)
    adapter.fit(params, Xtr, ytr, Xes, yes, seed) -> (model, info)
        model có predict_proba() và classes_ (nhãn gốc); info có 'iters', 'complexity'
    adapter.finalize(model) -> model  (việc cần làm trước khi đóng gói, vd XGBoost về CPU)
"""
import json, time, warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score, make_scorer
from sklearn.model_selection import ParameterSampler

from ids_features import decide, predict_frame
from ids_common import (WORK, cap_sample, choose_threshold, evaluate, evaluate_external,
                        frac_sample, full_proba, inference_time, load_data, log_run,
                        save_bundle, save_tables, xy)

ES_MOD = 10   # nhóm có group_id % 10 == 0 -> tập early stopping (~10%)


class Remapped:
    """Chỉ dùng NỘI BỘ khi mô hình được học trên một phần các lớp (Bảng B).
    Không bao giờ được đóng gói vào bundle."""

    def __init__(self, model, present):
        self.model, self.classes_ = model, np.asarray(present)

    def predict_proba(self, X):
        return self.model.predict_proba(X)

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]


def split_es(frame):
    es = (frame['group_id'].to_numpy() % ES_MOD) == 0
    return frame[~es], frame[es]


def fit_frame(adapter, params, frame, feats, seed):
    tr, es = split_es(frame)
    Xtr, ytr = xy(tr, feats)
    Xes, yes = xy(es, feats)
    t0 = time.time()
    model, info = adapter.fit(params, Xtr, ytr, Xes, yes, seed)
    info['fit_giay'] = round(time.time() - t0, 1)
    return model, info


def run(adapter, C):
    """C: dict cấu hình (xem cell cấu hình trong notebook)."""
    warnings.filterwarnings('ignore', category=UserWarning)
    T0 = time.time()
    el = lambda: f'{(time.time() - T0) / 60:.1f} phút'
    run_id = time.strftime('%Y%m%d-%H%M%S') + f'-{adapter.short}' + ('-quick' if C['QUICK'] else '')
    run_dir = WORK / 'runs' / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    SEED, MODEL = C['SEED'], adapter.name

    # ---------------------------------------------------------------- 1. dữ liệu
    df, fs, SHA = load_data()
    if C['QUICK']:
        df = frac_sample(df, 0.03, seed=SEED).reset_index(drop=True)
        print(f'QUICK: dùng {len(df):,} dòng')
    CLASSES = fs['classes']
    K, B = len(CLASSES), CLASSES.index('BENIGN')
    train_pool = df[df['fold'] != 0]
    test = df[df['fold'] == 0]          # chỉ chạm tới ở bước 4
    print(f'train (fold 1-4): {len(train_pool):,} | test (fold 0): {len(test):,}')

    # ---------------------------------------------------------------- 2. tinh chỉnh
    feats = fs[C['TUNE_FSET']]
    tune = cap_sample(train_pool, C['TUNE_CAP'], C['TUNE_BENIGN_CAP'], SEED)
    print(f'\nTinh chỉnh trên {len(tune):,} dòng, {C["N_ITER"]} cấu hình × 4 fold | {el()}')
    configs = list(ParameterSampler(C['SPACE'], n_iter=C['N_ITER'], random_state=SEED))
    rows = []
    for i, p in enumerate(configs):
        t0, scores, cx, its = time.time(), [], [], []
        for k in (1, 2, 3, 4):
            m, info = fit_frame(adapter, p, tune[tune.fold != k], feats, SEED)
            va = tune[tune.fold == k]
            Xva, yva = xy(va, feats)
            pred = decide(full_proba(m, Xva, K), B, None)
            scores.append(f1_score(yva, pred, labels=np.unique(yva), average='macro'))
            cx.append(info['complexity']); its.append(info['iters'])
        rows.append({'cfg': i, **{k: str(v) for k, v in p.items()},
                     'cv_macro_f1_mean': np.mean(scores), 'cv_macro_f1_std': np.std(scores),
                     'do_phuc_tap': float(np.mean(cx)), 'vong_hoac_epoch': float(np.mean(its)),
                     'giay': round(time.time() - t0, 1)})
        print(f'  cfg {i + 1}/{C["N_ITER"]}: macro-F1 {np.mean(scores):.4f} ± {np.std(scores):.4f}'
              f' | {np.mean(its):.0f} vòng/epoch | {time.time() - t0:.0f}s | {el()}')
    tune_df = pd.DataFrame(rows).sort_values('cv_macro_f1_mean', ascending=False)
    tune_df.to_csv(run_dir / 'tuning.csv', index=False, encoding='utf-8')
    # Quy tắc 1 độ lệch chuẩn: trong nhóm gần tốt nhất, chọn cấu hình ít phức tạp nhất.
    top = tune_df.iloc[0]
    near = tune_df[tune_df.cv_macro_f1_mean >= top.cv_macro_f1_mean - top.cv_macro_f1_std]
    best_row = near.sort_values('do_phuc_tap').iloc[0]
    BEST = configs[int(best_row.cfg)]
    print(f'\nChọn cfg {int(best_row.cfg)}: {BEST}')
    log_run(run_id, MODEL, C['TUNE_FSET'], 'tune_cv4_mau',
            {'macro_f1': round(float(best_row.cv_macro_f1_mean), 6),
             'macro_f1_std': round(float(best_row.cv_macro_f1_std), 6), 'n': len(tune)},
            params=BEST, sha=SHA)

    summary = {}
    for fset in C['FSETS']:
        feats = fs[fset]
        print(f'\n================ {fset} ({len(feats)} đặc trưng) | {el()} ================')
        assert 1 in C['FULL_CV'][fset], 'fold 1 phải có để chọn ngưỡng'

        # ------------------------------------------------------------ 3. CV đầy đủ
        cv_scores, thr = [], None
        for k in C['FULL_CV'][fset]:
            m, info = fit_frame(adapter, BEST, train_pool[train_pool.fold != k], feats, SEED)
            va = train_pool[train_pool.fold == k]
            P = full_proba(m, xy(va, feats)[0], K)
            sc, _ = evaluate(va['y'], P, decide(P, B, None), CLASSES, va, va['n_dup'])
            cv_scores.append(sc)
            log_run(run_id, MODEL, fset, f'cv_fold{k}', sc, BEST, SHA,
                    {'fit_giay': info['fit_giay'], 'vong_hoac_epoch': info['iters']})
            if k == 1:
                thr, thr_tab = choose_threshold(va['y'].to_numpy(), P, B, CLASSES, C['TARGET_FPR'])
                thr_tab.to_csv(run_dir / f'{fset}_nguong_validation.csv', index=False)
                print(f'  Ngưỡng chọn trên fold 1 (FPR <= {C["TARGET_FPR"]:.0%}): {thr:.4f}')
            del m, P
        if len(cv_scores) > 1:
            keys = [k for k in cv_scores[0] if isinstance(cv_scores[0][k], float)]
            agg = {k: round(float(np.mean([s[k] for s in cv_scores])), 6) for k in keys}
            agg.update({f'{k}_std': round(float(np.std([s[k] for s in cv_scores])), 6)
                        for k in ('macro_f1', 'fpr_benign', 'detection_rate')})
            log_run(run_id, MODEL, fset, f'cv{len(cv_scores)}_trung_binh', agg, BEST, SHA)

        # ------------------------------------------------------------ 4. mô hình cuối + test
        final, info = fit_frame(adapter, BEST, train_pool, feats, SEED)
        final = adapter.finalize(final)                # trước khi dự đoán, để test = triển khai
        Xte = xy(test, feats)[0]
        Pte = full_proba(final, Xte, K)
        timing = inference_time(final, Xte)
        extra = {'fit_giay': info['fit_giay'], 'vong_hoac_epoch': info['iters'], **timing}
        test_scores = None
        for stage, t in (('test_argmax', None), ('test_nguong', thr)):
            sc, tb = evaluate(test['y'], Pte, decide(Pte, B, t), CLASSES, test, test['n_dup'])
            log_run(run_id, MODEL, fset, stage, sc, {**BEST, 'threshold': t}, SHA, extra)
            save_tables(run_dir, f'{fset}_{stage}', tb)
            if stage == 'test_nguong':
                test_scores = sc
                print(tb['confusion'].to_string())
        pd.DataFrame(Pte, columns=[f'p_{c}' for c in CLASSES]).assign(
            y=test['y'].to_numpy(), row=test.index.to_numpy()).to_parquet(
            run_dir / f'{fset}_test_proba.parquet', index=False)

        # ------------------------------------------------------------ 5. mức quan trọng
        imp = pd.DataFrame({'dac_trung': feats})
        native = getattr(final, 'feature_importances_', None)
        if native is not None:
            imp['native_gain'] = native
        ps = cap_sample(test, C['PERM_CAP'], C['PERM_CAP'], SEED)
        Xp, yp = xy(ps, feats)
        pi = permutation_importance(final, Xp, yp, scoring=make_scorer(f1_score, average='macro'),
                                    n_repeats=3, random_state=SEED, n_jobs=1)
        imp['permutation_mean'], imp['permutation_std'] = pi.importances_mean, pi.importances_std
        imp = imp.sort_values('permutation_mean', ascending=False)
        imp.to_csv(run_dir / f'{fset}_feature_importance.csv', index=False, encoding='utf-8')
        print('  Top 10 permutation importance:\n', imp.head(10).to_string(index=False))

        # ------------------------------------------------------------ 6. đóng gói + kiểm tra
        path = WORK / 'models' / f'{adapter.short}_{fset}.joblib'
        save_bundle(path, final, feats, fset, CLASSES, thr, C['TARGET_FPR'], BEST, run_id, SHA,
                    test_scores, MODEL)
        mb = round(path.stat().st_size / 2**20, 2)
        re = joblib.load(path)
        out = predict_frame(re, test.head(2000)[feats])
        ref = np.asarray(CLASSES, dtype=object)[decide(Pte[:2000], B, thr)]
        assert (out['label'].to_numpy() == ref).all(), 'Bundle load lại cho kết quả khác!'
        print(f'  Đã lưu {path.name} ({mb} MB), load lại khớp 100%')
        summary[fset] = {'macro_f1': test_scores['macro_f1'],
                         'fpr_benign': test_scores['fpr_benign'], 'model_mb': mb, **timing}
        del final, Pte, Xte, re

        # ------------------------------------------------------------ 7. Bảng B
        if C['RUN_CROSS_DATASET']:
            for test_ds in ('2017', '2018', '2019'):
                tr = df[df.source_dataset != test_ds]
                te = df[df.source_dataset == test_ds]
                te = te[~te.group_id.isin(tr.group_id.unique())]
                te = te[te.y.isin(tr.y.unique())]
                tr = cap_sample(tr, C['CROSS_CAP'], C['CROSS_CAP'], SEED)
                m, info = fit_frame(adapter, BEST, tr, feats, SEED)
                P = full_proba(m, xy(te, feats)[0], K)
                sc, tb = evaluate(te['y'], P, decide(P, B, None), CLASSES, te)
                log_run(run_id, MODEL, fset, f'cheo_test{test_ds}', sc, BEST, SHA,
                        {'train_bo': '+'.join(sorted({'2017', '2018', '2019'} - {test_ds}))})
                save_tables(run_dir, f'{fset}_cheo_test{test_ds}', tb)
                del m, P

        # ------------------------------------------------------------ 8. Bảng C (lab)
        if C.get('LAB_CSV'):
            sc, tb = evaluate_external(joblib.load(path), C['LAB_CSV'], C['LAB_LABEL_COL'],
                                       C['LAB_LABEL_MAP'])
            log_run(run_id, MODEL, fset, 'lab', sc, BEST, SHA)
            save_tables(run_dir, f'{fset}_lab', tb)

    json.dump({'run_id': run_id, 'best_params': BEST, 'summary': summary, 'quick': C['QUICK'],
               'thoi_gian_phut': round((time.time() - T0) / 60, 1)},
              open(run_dir / 'summary.json', 'w', encoding='utf-8'), ensure_ascii=False,
              indent=2, default=str)
    print(f'\nXONG sau {el()}.')
    print(pd.DataFrame(summary).T)
    print(f'Kết quả: {WORK / "experiments.csv"}, {run_dir}, {WORK / "models"}')
    return run_id


## Cấu hình

In [ ]:
QUICK = True          # True: chạy thử ~3% dữ liệu để bắt lỗi. False: chạy thật (Save & Run All).

C = dict(
    QUICK=QUICK, SEED=42,
    FSETS=['F_NOOS', 'F_ALL'], TUNE_FSET='F_NOOS',
    FULL_CV={'F_NOOS': [1, 2, 3, 4], 'F_ALL': [1]},     # fold 1 luôn phải có (chọn ngưỡng)
    N_ITER=12, TUNE_CAP=30_000, TUNE_BENIGN_CAP=100_000,
    CROSS_CAP=300_000, PERM_CAP=20_000,
    TARGET_FPR=0.01, RUN_CROSS_DATASET=True,
    # Dữ liệu lab (Bảng C): điền khi có CSV CICFlowMeter từ lab
    LAB_CSV=None, LAB_LABEL_COL='Label',
    LAB_LABEL_MAP={'BENIGN': 'BENIGN', 'nmap': 'PortScan', 'hydra': 'BruteForce',
                   'hping3': 'DoS/DDoS'},
    SPACE={
        'num_leaves': [31, 63, 127, 255],
        'learning_rate': [0.05, 0.1],
        'min_child_samples': [20, 50, 100],
        'colsample_bytree': [0.6, 0.8, 1.0],
        'subsample': [0.7, 0.85, 1.0],
        'reg_lambda': [0, 1, 5],
        'cw_power': [0, 0.5, 1],       # trọng số lớp = (cân bằng)^cw_power; 0 = không cân bằng
    },
)
if QUICK:
    C.update(N_ITER=2, TUNE_CAP=3_000, TUNE_BENIGN_CAP=10_000, CROSS_CAP=20_000, PERM_CAP=1_000)

MAX_ROUNDS, ES_ROUNDS = (50, 10) if QUICK else (3000, 50)


## Adapter LightGBM
Phần riêng của mô hình này; mọi bước còn lại nằm trong `ids_runner.run`.

In [ ]:
import warnings
import numpy as np
import lightgbm as lgb
from ids_runner import run

# LightGBM >= 4.6 báo eval_set sắp bỏ; vẫn dùng eval_set để chạy được trên cả bản cũ.
warnings.filterwarnings('ignore', message='.*eval_set.*deprecated.*')

print('LightGBM', lgb.__version__)


def class_weights(y, power):
    """Trọng số từng lớp = (n / (k * n_c)) ** power, tính trên tập huấn luyện."""
    present, counts = np.unique(y, return_counts=True)
    return present, (len(y) / (len(present) * counts)) ** power


class LGBMAdapter:
    name, short = 'LightGBM', 'lgbm'

    def fit(self, params, Xtr, ytr, Xes, yes, seed):
        p = dict(params)
        present, cw = class_weights(ytr, p.pop('cw_power'))
        keep = np.isin(yes, present)
        Xes, yes = Xes[keep], yes[keep]
        m = lgb.LGBMClassifier(n_estimators=MAX_ROUNDS, subsample_freq=1, importance_type='gain',
                               random_state=seed, n_jobs=-1, verbose=-1, **p)
        m.fit(Xtr, ytr, sample_weight=cw[np.searchsorted(present, ytr)],
              eval_set=[(Xes, yes)], eval_sample_weight=[cw[np.searchsorted(present, yes)]],
              callbacks=[lgb.early_stopping(ES_ROUNDS, verbose=False)])
        iters = int(m.best_iteration_ or MAX_ROUNDS)
        return m, {'iters': iters, 'complexity': iters * p['num_leaves']}

    def finalize(self, model):
        return model


## Chạy

In [ ]:
run_id = run(LGBMAdapter(), C)
